# Neural Collaborative Filtering (NCF) in Google Colab

This notebook walks through the full pipeline for the **NCF** movie recommender system.

> **Runtime tip:** Go to `Runtime → Change runtime type` and select **GPU (T4)** before running. This will train the network faster. If a GPU is not available, you can still use a CPU environment but the training will be slower.

**Reference paper:** He et al., [Neural Collaborative Filtering](https://arxiv.org/abs/1708.05031), WWW 2017.

## Preparation

In [ ]:
import os
import sys
import time
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torch.backends.cudnn as cudnn
import torch.nn.functional as F

from collections import defaultdict





Check GPU availability. Training on CPU is also possible but slower.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Training will be slower but possible.')

In [ ]:
# Make sure we are on Google's colab root directory

%cd /content
!ls

In [ ]:
# Optionally remove a previous version of the repository (uncomment the line below and execute)
#!rm -rf NCF-Fork/

## 1. Clone Repository & Install Dependencies

We clone the project directly from GitHub so all source modules (`model.py`, `data_utils.py`, `evaluate.py`, `config.py`, `preprocess_data.py`) are available.

In [ ]:
REPO_URL  = 'https://github.com/noe-tec/NCF-Fork.git'
REPO_NAME = 'NCF-Fork'

if not os.path.isdir(REPO_NAME):
    !git clone {REPO_URL}
else:
    print(f'Repository already cloned at ./{REPO_NAME}')

%cd {REPO_NAME}

In [ ]:
!ls

In [ ]:
# Install required packages (if not available)
# tensorboardX is imported by main.py; scipy and pandas are used by data_utils
#!pip install -q tensorboardX scipy pandas numpy

In [ ]:
REPO_DIR = os.getcwd()
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Smoke-test that all local modules are importable
import config, data_utils, evaluate, preprocess_data
from preprocess_data import build_dataset

print('All modules imported successfully.')
print(f'Working directory: {REPO_DIR}')

## 2. Download & Preprocess the MovieLens 100K Dataset

The raw dataset is downloaded from the GroupLens servers, extracted, and then transformed into the NCF-ready format:

- **Leave-one-out split** — the most recent interaction per user goes to the test set.
- **Negative sampling** — 99 unobserved items are sampled per test user for evaluation.
- **ID remapping** — original `userId` / `movieId` values are remapped to contiguous indices starting from 0.

Output files are written to `data/processed/ml-100k-custom/`.

In [ ]:


RAW_DIR = Path('data/raw/ml-100k')
ZIP_URL = 'https://files.grouplens.org/datasets/movielens/ml-100k.zip'
ZIP_PATH = Path('data/raw/ml-100k.zip')

RAW_DIR.parent.mkdir(parents=True, exist_ok=True)

if not RAW_DIR.exists():
    print('Downloading MovieLens 100K...')
    urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
    print('Extracting...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall('data/raw/')
    print(f'Done. Files at: {RAW_DIR}')
else:
    print(f'Raw data already present at: {RAW_DIR}')

# Verify the two files that preprocess_data.py needs
assert (RAW_DIR / 'u.data').exists(), 'u.data not found'
assert (RAW_DIR / 'u.item').exists(), 'u.item not found'
print('u.data and u.item found.')

In [ ]:
PROCESSED_DIR = Path('data/processed/ml-100k-custom')
TRAIN_FILE    = PROCESSED_DIR / 'ml-100k.train.rating'

if not TRAIN_FILE.exists():
    print('Preprocessing dataset...')
    build_dataset(
        dataset='ml-100k',
        raw_dir=RAW_DIR,
        output_dir=PROCESSED_DIR,
        min_rating=None,   # treat every observed rating as positive
        num_negatives=99,
        seed=42,
    )
else:
    print(f'Processed files already exist at: {PROCESSED_DIR}')

# List generated files
for f in sorted(PROCESSED_DIR.iterdir()):
    print(f'  {f.name}')

In [ ]:
!head data/processed/ml-100k-custom/ml-100k.train.rating
# user_idx , item_idx , rating , timestamp

## 3. Train the NeuMF Model

We train **NeuMF** which combines:

- **GMF** — Generalized Matrix Factorization
- **MLP** — Multi-Layer Perceptron

The output of both paths is concatenated and fed to a single prediction layer.

Evaluation uses **Hit Rate @ 10** (HR@10) and **NDCG@10** on the leave-one-out test set.

**Answer the following questions:**

1.   What problem does NCF aim to solve?
2.   How are HitRate@10 and NDCG@10 calculated?


#### Define dataset class

In [ ]:
class NCFData(data.Dataset):
	def __init__(self, features,
				num_item, train_mat=None, num_ng=0, is_training=None):
		super(NCFData, self).__init__()
		""" Note that the labels are only useful when training, we thus
			add them in the ng_sample() function.
		"""
		self.features_ps = features
		self.num_item = num_item
		self.train_mat = train_mat
		self.num_ng = num_ng
		self.is_training = is_training
		self.labels = [0 for _ in range(len(features))]

	def ng_sample(self):
		assert self.is_training, 'no need to sampling when testing'

		self.features_ng = []
		for x in self.features_ps:
			u = x[0]
			for t in range(self.num_ng):
				j = np.random.randint(self.num_item)
				while (u, j) in self.train_mat:
					j = np.random.randint(self.num_item)
				self.features_ng.append([u, j])

		labels_ps = [1 for _ in range(len(self.features_ps))]
		labels_ng = [0 for _ in range(len(self.features_ng))]

		self.features_fill = self.features_ps + self.features_ng
		self.labels_fill = labels_ps + labels_ng

	def __len__(self):
		return (self.num_ng + 1) * len(self.labels)

	def __getitem__(self, idx):
		features = self.features_fill if self.is_training \
					else self.features_ps
		labels = self.labels_fill if self.is_training \
					else self.labels

		user = features[idx][0]
		item = features[idx][1]
		label = labels[idx]
		return user, item ,label


#### Define hyperparameters and create dataloaders

In [ ]:
import config
import evaluate
import data_utils

# ── Hyperparameters ───────────────────────────────────────────────
LR          = 0.001
DROPOUT     = 0.0
BATCH_SIZE  = 256
EPOCHS      = 20
TOP_K       = 10
FACTOR_NUM  = 32
NUM_LAYERS  = 3
NUM_NG      = 4      # negative samples per positive interaction during training
TEST_NUM_NG = 99     # must match num_negatives used in preprocessing

if torch.cuda.is_available():
    cudnn.benchmark = True

# ── Load data ────────────────────────────────────────────────────
train_data, test_data, user_num, item_num, train_mat = data_utils.load_all()

train_dataset = NCFData(train_data, item_num, train_mat, NUM_NG, True)
test_dataset  = NCFData(test_data,  item_num, train_mat, 0,      False)

train_loader = data.DataLoader(train_dataset,
                               batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader  = data.DataLoader(test_dataset,
                               batch_size=TEST_NUM_NG + 1, shuffle=False, num_workers=0)

print(f'Users: {user_num}  |  Items: {item_num}')
print(f'Train interactions: {len(train_data):,}')
print(f'Test  interactions: {len(test_data):,}')

#### Show the first 10 samples of a batch

In [ ]:
train_loader.dataset.ng_sample()
users, items, labels = next(iter(train_loader))

print("users: ", users[:10])
print("items: ", items[:10])
print("labels:", labels[:10])
print(f"\nBatch size: {len(labels)}")
print(f"Positives in the batch: {labels.sum().item()}")
print(f"Negatives in the batch: {(labels == 0).sum().item()}")

**Answer the following questions:**

1.   What does negative sampling do (ng_sample)?
2.   What are the posible values of label?
3.   Why are the original rating values not return in the __getitem__ method?




#### Define the model's architecture


In [ ]:
class NCF(nn.Module):
	def __init__(self, user_num, item_num, factor_num, num_layers,
					dropout, model, GMF_model=None, MLP_model=None):
		super(NCF, self).__init__()
		"""
		user_num: number of users;
		item_num: number of items;
		factor_num: number of predictive factors;
		num_layers: the number of layers in MLP model;
		dropout: dropout rate between fully connected layers;
		model: 'MLP', 'GMF', 'NeuMF-end', and 'NeuMF-pre';
		GMF_model: pre-trained GMF weights;
		MLP_model: pre-trained MLP weights.
		"""
		self.dropout = dropout
		self.model = model
		self.GMF_model = GMF_model
		self.MLP_model = MLP_model

		self.embed_user_GMF = nn.Embedding(user_num, factor_num)
		self.embed_item_GMF = nn.Embedding(item_num, factor_num)
		self.embed_user_MLP = nn.Embedding(
				user_num, factor_num * (2 ** (num_layers - 1)))
		self.embed_item_MLP = nn.Embedding(
				item_num, factor_num * (2 ** (num_layers - 1)))

		MLP_modules = []
		for i in range(num_layers):
			input_size = factor_num * (2 ** (num_layers - i))
			MLP_modules.append(nn.Dropout(p=self.dropout))
			MLP_modules.append(nn.Linear(input_size, input_size//2))
			MLP_modules.append(nn.ReLU())
		self.MLP_layers = nn.Sequential(*MLP_modules)

		if self.model in ['MLP', 'GMF']:
			predict_size = factor_num
		else:
			predict_size = factor_num * 2
		self.predict_layer = nn.Linear(predict_size, 1)

		self._init_weight_()

	def _init_weight_(self):
		""" We leave the weights initialization here. """
		if not self.model == 'NeuMF-pre':
			nn.init.normal_(self.embed_user_GMF.weight, std=0.01)
			nn.init.normal_(self.embed_user_MLP.weight, std=0.01)
			nn.init.normal_(self.embed_item_GMF.weight, std=0.01)
			nn.init.normal_(self.embed_item_MLP.weight, std=0.01)

			for m in self.MLP_layers:
				if isinstance(m, nn.Linear):
					nn.init.xavier_uniform_(m.weight)
			nn.init.kaiming_uniform_(self.predict_layer.weight,
									a=1, nonlinearity='sigmoid')

			for m in self.modules():
				if isinstance(m, nn.Linear) and m.bias is not None:
					m.bias.data.zero_()
		else:
			# embedding layers
			self.embed_user_GMF.weight.data.copy_(
							self.GMF_model.embed_user_GMF.weight)
			self.embed_item_GMF.weight.data.copy_(
							self.GMF_model.embed_item_GMF.weight)
			self.embed_user_MLP.weight.data.copy_(
							self.MLP_model.embed_user_MLP.weight)
			self.embed_item_MLP.weight.data.copy_(
							self.MLP_model.embed_item_MLP.weight)

			# mlp layers
			for (m1, m2) in zip(
				self.MLP_layers, self.MLP_model.MLP_layers):
				if isinstance(m1, nn.Linear) and isinstance(m2, nn.Linear):
					m1.weight.data.copy_(m2.weight)
					m1.bias.data.copy_(m2.bias)

			# predict layers
			predict_weight = torch.cat([
				self.GMF_model.predict_layer.weight,
				self.MLP_model.predict_layer.weight], dim=1)
			precit_bias = self.GMF_model.predict_layer.bias + \
						self.MLP_model.predict_layer.bias

			self.predict_layer.weight.data.copy_(0.5 * predict_weight)
			self.predict_layer.bias.data.copy_(0.5 * precit_bias)

	def forward(self, user, item):
		if not self.model == 'MLP':
			embed_user_GMF = self.embed_user_GMF(user)
			embed_item_GMF = self.embed_item_GMF(item)
			output_GMF = embed_user_GMF * embed_item_GMF
		if not self.model == 'GMF':
			embed_user_MLP = self.embed_user_MLP(user)
			embed_item_MLP = self.embed_item_MLP(item)
			interaction = torch.cat((embed_user_MLP, embed_item_MLP), -1)
			output_MLP = self.MLP_layers(interaction)

		if self.model == 'GMF':
			concat = output_GMF
		elif self.model == 'MLP':
			concat = output_MLP
		else:
			concat = torch.cat((output_GMF, output_MLP), -1)

		prediction = self.predict_layer(concat)
		return prediction.view(-1)


In [ ]:
# ── Build model ───────────────────────────────────────────────────
ncf_model = NCF(
    user_num=user_num,
    item_num=item_num,
    factor_num=FACTOR_NUM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    model=config.model,
    GMF_model=None,
    MLP_model=None,
)
ncf_model.to(device)

loss_fn   = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(ncf_model.parameters(), lr=LR)

os.makedirs(config.model_path, exist_ok=True)

print(f'Model architecture: {config.model}')
total_params = sum(p.numel() for p in ncf_model.parameters())
print(f'Trainable parameters: {total_params:,}')

**Answer the following questions:**

1.   Does the model use any form of regularization? If so, is it enabled?
2.   What activation function does the model use?
3.   According to the paper, the model output uses a sigmoid activation function. Where is this implemented in the code?
4.   What loss function does the model use?
5.   Is the learning rate fixed or variable?
6.   Explain the function of the nn.Embedding module

In [ ]:
# A copy of the model is temporaly saved in /content/NCF-Fork/models
# To make it persistant, we need to mount google drive

from google.colab import drive
drive.mount('/content/drive')
model_dir = "/content/drive/MyDrive/modelos_pytorch"
os.makedirs(model_dir, exist_ok=True)
persistent_model_path = f"/content/drive/MyDrive/modelos_pytorch/{config.model}.pth"

In [ ]:
if os.path.exists(persistent_model_path):
    answer = input(
        f'A saved model already exists at:\n{persistent_model_path}\n\n'
        'Do you really want to continue training and overwrite it? [y/N]: '
    ).strip().lower()

    if answer not in ['y', 'yes']:
        raise SystemExit('Training cancelled. Existing model was not overwritten.')

# ── Training loop ─────────────────────────────────────────────────
best_hr, best_ndcg, best_epoch = 0.0, 0.0, 0

for epoch in range(EPOCHS):
    ncf_model.train()
    t0 = time.time()

    train_loader.dataset.ng_sample()  # resample negatives each epoch

    for user, item, label in train_loader:
        user  = user.to(device)
        item  = item.to(device)
        label = label.float().to(device)

        optimizer.zero_grad()
        prediction = ncf_model(user, item)
        loss = loss_fn(prediction, label)
        loss.backward()
        optimizer.step()

    ncf_model.eval()
    HR, NDCG = evaluate.metrics(ncf_model, test_loader, TOP_K, device)
    elapsed  = time.strftime('%M:%S', time.gmtime(time.time() - t0))

    print(f'Epoch {epoch+1:02d}/{EPOCHS}  [{elapsed}]  '
          f'HR@{TOP_K}: {HR:.4f}  NDCG@{TOP_K}: {NDCG:.4f}')

    if HR > best_hr:
        best_hr, best_ndcg, best_epoch = HR, NDCG, epoch + 1
        torch.save(ncf_model, persistent_model_path)

print(f'\nBest model at epoch {best_epoch}: '
      f'HR@{TOP_K} = {best_hr:.4f},  NDCG@{TOP_K} = {best_ndcg:.4f}')
print(f'Saved to: {persistent_model_path}')

In [ ]:
!ls /content/drive/MyDrive/modelos_pytorch

# Load a model from drive


**Answer the following questions:**

1. What optimizer are we using?
2. What is the function of optimizer.zero_grad()?
2. What is the function of loss.backward()?
3. What is the function of optimizer.step()?
4. Are the performance metrics comparable to those reported in the paper? Explain your answer and discuss potential reasons for any differences.

## 4. Load Model & Data for Inference

We reload the best checkpoint saved during training along with:
- The training rating file (to know each user's interaction history)
- The movie mapping CSV (to resolve internal `item_idx` → movie title)

In [ ]:
DATASET      = config.dataset          # 'ml-100k'
TRAIN_PATH   = config.train_rating
TEST_PATH    = config.test_rating
MAPPING_PATH = f'data/processed/{DATASET}-custom/{DATASET}.movie_mapping.csv'
MODEL_PATH   = persistent_model_path
#MODEL_PATH  = f'{config.model_path}{config.model}.pth'  # temporal model path

# ── Movie mapping: item_idx → title ──────────────────────────────
mapping_df   = pd.read_csv(MAPPING_PATH)
id_to_title  = dict(zip(mapping_df['item_idx'], mapping_df['title']))
id_to_movieid = dict(zip(mapping_df['item_idx'], mapping_df['movieId']))

# ── Training interactions ─────────────────────────────────────────
train_df = pd.read_csv(TRAIN_PATH, sep='\t', header=None,
                       names=['user', 'item', 'rating', 'ts'],
                       dtype={'user': 'int32', 'item': 'int32'})

all_item_ids = sorted(train_df['item'].unique().tolist())
all_user_ids = sorted(train_df['user'].unique().tolist())

print(f'Users       : {len(all_user_ids)}')
print(f'Items       : {len(all_item_ids)}')
print(f'Interactions: {len(train_df):,}')

# ── Load best checkpoint ──────────────────────────────────────────
try:
    ncf = torch.load(MODEL_PATH, map_location=device, weights_only=False)
except TypeError:
    ncf = torch.load(MODEL_PATH, map_location=device)

ncf.eval()
print(f'\nModel loaded: {type(ncf).__name__} from {MODEL_PATH}')

In [ ]:
# ── Shared helper functions ───────────────────────────────────────

def get_user_history(uid, train_df, id_to_title):
    """Return a list of (item_idx, title) for all movies a user interacted with."""
    items = train_df[train_df['user'] == uid]['item'].values
    return [(int(iid), id_to_title.get(iid, f'Unknown ({iid})')) for iid in items]


def get_ncf_recommendations(uid, model, train_df, all_item_ids, device, top_n=10):
    """Score all unseen items for a user and return the top-N ranked list."""
    model.eval()
    seen   = set(train_df[train_df['user'] == uid]['item'].values)
    unseen = [i for i in all_item_ids if i not in seen]
    if not unseen:
        return []
    u_t = torch.tensor([uid] * len(unseen), dtype=torch.long).to(device)
    i_t = torch.tensor(unseen, dtype=torch.long).to(device)
    with torch.no_grad():
        scores = model(u_t, i_t).cpu().numpy()
    ranked = sorted(zip(unseen, scores.tolist()), key=lambda x: x[1], reverse=True)
    return [
        (iid, id_to_title.get(iid, f'Unknown ({iid})'), score)
        for iid, score in ranked[:top_n]
    ]


def search_movies(query, id_to_title, n=10):
    """Search movies by substring and return a DataFrame with item_id and title."""
    results = [(iid, title) for iid, title in id_to_title.items()
               if query.lower() in title.lower()]
    results.sort(key=lambda x: x[1])
    return pd.DataFrame(results[:n], columns=['item_id', 'title'])

**Answer the following questions:**

1. Why do we have to call model.eval() before performing inference?
2. Why is it important to perform inference inside a torch.no_grad() block?
3. Explain the general steps performed inside the get_ncf_recommendations function

## 5. Inference: Existing User

Pick any `USER_ID` in the range `0 – 942`.  
The model scores every movie the user has **not** seen and ranks them by predicted interaction probability.

In [ ]:
USER_ID = 699  # change to any user index (0–942)

history = get_user_history(USER_ID, train_df, id_to_title)
print(f'User {USER_ID} — {len(history)} interactions in training data')
print()
print('Sample interaction history (first 20):')
for i, (iid, title) in enumerate(history[:20], 1):
    print(f'  {i:2d}. [{iid:4d}] {title}')
if len(history) > 20:
    print(f'  ... and {len(history) - 20} more')

In [ ]:
TOP_N = 10

recs = get_ncf_recommendations(USER_ID, ncf, train_df, all_item_ids, device, top_n=TOP_N)

print(f'Top-{TOP_N} recommendations for User {USER_ID}:')
print()
print(f'  {"#":>2}  {"item_idx":>8}  {"movieId":>8}  {"score":>7}  title')
print('  ' + '-' * 62)
for rank, (iid, title, score) in enumerate(recs, 1):
    mid = id_to_movieid.get(iid, '?')
    print(f'  {rank:>2}.  {iid:>8}  {mid:>8}  {score:>7.4f}  {title}')

**Answer the following questions:**
1. What do the recommendation scores represent?
2. Are these scores limited to the range from 0 to 1? Explain why or why not.
3. How can these scores be converted into probabilities of interaction?
4. Do you think the recommender system is producing reasonable recommendations? Justify your answer using specific examples from the results.

## 6. Cold-Start Inference: New User

In real-world recommendation systems, new users usually do not have any previous interaction history. This makes it challenging to generate personalized recommendations with standard collaborative filtering methods, which rely on past user-item interactions. This is known as the cold-start problem.

In this section, the goal is to design and evaluate a mechanism that uses the previously trained NCF model to recommend movies to a new user.

1. Propose and implement a cold-start recommendation mechanism for new users. The mechanism should generate a list of 10 movie recommendations for a user that does not exist in the system. Assume that the new user provides a list of at least 20 movies they like, all of which must belong to the training set.

2. To evaluate the proposed approach, each team member must define a personal list of 20 movies they like from the training set. This list will be used as input to the cold-start mechanism, which should then produce 10 movie recommendations for that user.

3. The team must briefly explain the rationale behind the proposed cold-start strategy, including how the provided liked movies are used to generate recommendations and how already-liked movies are excluded from the final recommendation list.

**In this cell explain and justify your team's cold-start proposal**

**Proposal:**


**Justification:**

In [ ]:
# Team's implementation of the proposal.
# Your code here

In [ ]:
# Team member 1 - Favourite movies and system recommendations
# After generating the recommendations, answer the following question:
# Are the recommendations reasonable for you? Briefly justify your answer.

In [ ]:
# Team member 2 - Favourite movies and system recommendations
# After generating the recommendations, answer the following question:
# Are the recommendations reasonable for you? Briefly justify your answer.

In [ ]:
# Team member 3 - Favourite movies and system recommendations
# After generating the recommendations, answer the following question:
# Are the recommendations reasonable for you? Briefly justify your answer.

In [ ]:
# Team member 4 - Favourite movies and system recommendations
# After generating the recommendations, answer the following question:
# Are the recommendations reasonable for you? Briefly justify your answer.

**Challenge (Optional)**

Note: "This challenge requires significant modifications across the full pipeline and is intended for teams that have completed all previous sections with time to spare. If you feel like trying it create a backup copy of your code before modifying anything."

Extend the current architecture so that it predicts user ratings in addition to interaction probabilities. In other words, transform the architecture into a multi-task learning model.

This extension will require modifications across the full pipeline, including the model architecture, dataset preparation, training objective, and evaluation functions. Specifically, the model should learn to perform two related tasks simultaneously:

Predict whether a user is likely to interact with an item.
Predict the rating that the user would assign to that item.

To complete this challenge, consider the following adaptations:

Model architecture: Add an additional prediction head for rating estimation, while keeping the existing interaction-probability output.
Dataset pipeline: Ensure that each training example includes both the interaction label and the corresponding rating value.
Training loss: Combine the interaction loss and the rating prediction loss into a single multi-task objective, using appropriate weighting if necessary.
Evaluation: Report separate metrics for each task, such as classification metrics for interaction prediction and regression metrics for rating prediction.

The goal is to design a model that can jointly learn from implicit feedback and explicit rating information, improving its ability to capture user preferences.